# Data Engineering — Moduł 4: dbt — podstawy

**dbt** (*data build tool*) to narzędzie do transformacji danych, które WYKONUJE się w
duchu ELT poznanym w Module 1: dane surowe są już w hurtowni, a dbt przekształca je
CZYSTYM SQL-em, wewnątrz samej hurtowni -- bez kopiowania danych na zewnętrzny serwer. W
tym module budujemy pierwsze modele dbt na naszym znanym zestawie danych sklepu
internetowego, ładowanym do lekkiej bazy analitycznej **DuckDB**.

## Spis treści
1. [Struktura projektu dbt](#sec1)
2. [Pierwszy model i `source()`](#sec2)
3. [`ref()` — łączenie modeli w łańcuch](#sec3)
4. [Materializacje: view, table, incremental, ephemeral](#sec4)
5. [`dbt run`, `dbt build`, `dbt compile`](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Generujemy nasz znany zestaw danych sklepu internetowego i ładujemy go do **DuckDB** (lekka, plikowa baza analityczna) jako schemat `raw` -- symulując surowe dane, które w prawdziwym projekcie załadowałby zewnętrzny proces ingestii (Moduł 1). Budujemy też szkielet projektu dbt (`dbt_project.yml`, `profiles.yml`, `sources.yml`) wskazujący na te dane.

In [ ]:
import os
import subprocess
import duckdb
import numpy as np
import pandas as pd

# --- 1. Generujemy surowe dane (ten sam generator co w kursie Databricks/PySpark) ---
rng = np.random.default_rng(7)

n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=[0.35, 0.45, 0.20]),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}
kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2) for k in kategorie_produktow],
})

n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values
prawdop_anulowania = np.where(segment_zamowien == "Nowy", 0.22, np.where(segment_zamowien == "Standardowy", 0.10, 0.04))
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(anulowane == 1, "Anulowane", rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]))
data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)
zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]
for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id, "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id), "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]), "rabat": float(rabat),
        })
        pozycja_id += 1
pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- 2. Ladujemy surowe dane do DuckDB, schemat "raw" (jakby zaladowal je zewnetrzny proces ingestii) ---
DBT_DIR = os.path.abspath("dbt_kurs")
os.makedirs(f"{DBT_DIR}/models", exist_ok=True)
DUCKDB_PATH = f"{DBT_DIR}/kurs_de.duckdb"

con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.klienci AS SELECT * FROM klienci_pd")
con.execute("CREATE OR REPLACE TABLE raw.produkty AS SELECT * FROM produkty_pd")
con.execute("CREATE OR REPLACE TABLE raw.zamowienia AS SELECT * FROM zamowienia_pd")
con.execute("CREATE OR REPLACE TABLE raw.pozycje_zamowien AS SELECT * FROM pozycje_zamowien_pd")
con.close()

# --- 3. Szkielet projektu dbt ---
with open(f"{DBT_DIR}/dbt_project.yml", "w") as f:
    f.write(
        "name: kurs_de\n"
        "version: '1.0'\n"
        "profile: kurs_de\n"
        "model-paths: [\"models\"]\n"
        "seed-paths: [\"seeds\"]\n"
        "snapshot-paths: [\"snapshots\"]\n"
    )

with open(f"{DBT_DIR}/profiles.yml", "w") as f:
    f.write(
        "kurs_de:\n"
        "  target: dev\n"
        "  outputs:\n"
        "    dev:\n"
        "      type: duckdb\n"
        f"      path: {DUCKDB_PATH}\n"
    )

with open(f"{DBT_DIR}/models/sources.yml", "w") as f:
    f.write(
        "version: 2\n"
        "sources:\n"
        "  - name: raw_dane\n"
        "    schema: raw\n"
        "    tables:\n"
        "      - name: klienci\n"
        "      - name: produkty\n"
        "      - name: zamowienia\n"
        "      - name: pozycje_zamowien\n"
    )

env = os.environ.copy()
env["DBT_PROFILES_DIR"] = DBT_DIR


def dbt(*args, check=True):
    """Uruchamia komende `dbt ...` w naszym projekcie i wypisuje wynik."""
    wynik = subprocess.run(
        ["dbt", *args, "--project-dir", DBT_DIR],
        env=env, capture_output=True, text=True,
    )
    print(wynik.stdout[-4000:])
    if wynik.returncode != 0:
        print(wynik.stderr[-2000:])
        if check:
            raise RuntimeError(f"dbt {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


print(f"Zaladowano surowe dane do {DUCKDB_PATH} (schemat raw).")
print(f"Projekt dbt: {DBT_DIR}")


<a id="sec1"></a>
## 1. Struktura projektu dbt

Komórka powyżej zbudowała za nas standardowy szkielet projektu dbt:

- **`dbt_project.yml`** -- konfiguracja projektu: nazwa, gdzie szukać modeli
  (`model-paths`), profil połączenia do użycia.
- **`profiles.yml`** -- dane połączenia do hurtowni (u nas: ścieżka do pliku DuckDB; w
  prawdziwym projekcie: konto i hasło do Snowflake/BigQuery/Databricks). W praktyce ten
  plik NIE trafia do repozytorium Git (zawiera sekrety) -- żyje osobno na maszynie każdego
  developera.
- **`models/`** -- folder z plikami `.sql`, każdy plik to JEDEN model (jedna tabela/widok
  w hurtowni).
- **`models/sources.yml`** -- deklaracja SUROWYCH tabel, które JUŻ ISTNIEJĄ w hurtowni
  (załadowane przez zewnętrzny proces ingestii), a którymi dbt się tylko POSŁUGUJE --
  nigdy ich nie tworzy ani nie modyfikuje.

> 🏭 **DuckDB w tym kursie, Snowflake/BigQuery/Databricks na produkcji**
>
> Uczymy się dbt na DuckDB, bo działa w pełni lokalnie, bez żadnej zewnętrznej infrastruktury -- ale modele SQL, które piszemy, są niemal identyczne z tym, co napisałbyś dla Snowflake, BigQuery czy Databricks. Zmienia się tylko `profiles.yml` (inny `type` i dane logowania) -- kod modeli zostaje ten sam. To jedna z głównych zalet dbt: przenośność między hurtowniami.

<a id="sec2"></a>
## 2. Pierwszy model i `source()`

Model dbt to zwykły plik `.sql` z JEDNYM zapytaniem `SELECT` -- BEZ `CREATE TABLE` czy
`INSERT`, o to dbt dba sam, na podstawie NAZWY PLIKU (nazwa pliku = nazwa
tabeli/widoku w hurtowni). Funkcja `source('nazwa_zrodla', 'nazwa_tabeli')` odwołuje się
do surowej tabeli zadeklarowanej w `sources.yml` -- to jedyny poprawny sposób
odwoływania się do danych spoza dbt.

> ⚡ **Dialekt SQL zależy od silnika -- nawet w dbt**
>
> dbt ujednolica SPOSÓB pisania modeli (`ref()`, `source()`, materializacje), ale NIE ujednolica dialektu SQL samego silnika -- funkcja `initcap()` (Postgres, Spark SQL z poprzedniego kursu) NIE istnieje w DuckDB, dlatego "Wielka Litera Na Początku" budujemy ręcznie z `upper()`/`substr()`. Przechodząc na inną hurtownię, czasem trzeba dostosować pojedyncze funkcje SQL -- reszta modelu (struktura, `ref()`) zostaje bez zmian.

In [ ]:
model_kod = '''
select
    klient_id,
    upper(left(miasto, 1)) || lower(substr(miasto, 2)) as miasto,
    coalesce(segment, 'Standardowy') as segment,
    dni_od_rejestracji
from {{ source('raw_dane', 'klienci') }}
'''

with open(f"{DBT_DIR}/models/stg_klienci.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_klienci")


In [ ]:
import duckdb

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.stg_klienci LIMIT 5").fetchdf())
con.close()


> ⚡ **Nazwa pliku = nazwa modelu w hurtowni**
>
> Plik `stg_klienci.sql` tworzy widok (domyślna materializacja) o nazwie `main.stg_klienci` w DuckDB -- prefiks `stg_` (od *staging*) to POWSZECHNA KONWENCJA w dbt na oznaczenie modeli, które tylko czyszczą surowe dane (tak jak silver w architekturze Medallion z kursu Databricks), a nie liczą jeszcze żadnej właściwej logiki biznesowej.

<a id="sec3"></a>
## 3. `ref()` — łączenie modeli w łańcuch

Funkcja **`ref('nazwa_modelu')`** odwołuje się do INNEGO MODELU dbt (nie do surowej
tabeli). To kluczowa różnica względem `source()`: `ref()` tworzy jawną ZALEŻNOŚĆ między
modelami -- dbt automatycznie buduje graf (DAG, dokładnie jak w Airflow!) i URUCHAMIA
modele we WŁAŚCIWEJ kolejności, nawet jeśli sam o to nie poprosisz.

In [ ]:
model_kod = '''
select
    klient_id,
    upper(left(miasto, 1)) || lower(substr(miasto, 2)) as miasto,
    coalesce(segment, 'Standardowy') as segment,
    dni_od_rejestracji
from {{ source('raw_dane', 'klienci') }}
'''
with open(f"{DBT_DIR}/models/stg_klienci.sql", "w") as f:
    f.write(model_kod)

model_kod = '''
select
    zamowienie_id,
    klient_id,
    data_zamowienia,
    status
from {{ source('raw_dane', 'zamowienia') }}
'''
with open(f"{DBT_DIR}/models/stg_zamowienia.sql", "w") as f:
    f.write(model_kod)

# Ten model UZYWA dwoch powyzszych przez ref() -- NIE przez source()!
model_kod = '''
select
    z.zamowienie_id,
    z.klient_id,
    k.miasto,
    k.segment,
    z.status,
    z.data_zamowienia
from {{ ref('stg_zamowienia') }} z
inner join {{ ref('stg_klienci') }} k on z.klient_id = k.klient_id
'''
with open(f"{DBT_DIR}/models/zamowienia_wzbogacone.sql", "w") as f:
    f.write(model_kod)

# dbt sam wykrywa kolejnosc: stg_klienci i stg_zamowienia PRZED zamowienia_wzbogacone
dbt("run")


In [ ]:
con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.zamowienia_wzbogacone LIMIT 5").fetchdf())
con.close()


> ⚠️ **Nigdy nie pisz nazwy tabeli "na sztywno"**
>
> Częsty błąd początkujących: napisanie `from main.stg_klienci` zamiast `from {{ ref('stg_klienci') }}`. Działa to dopóty, dopóki nie zmienisz schematu, docelowej bazy (`dev`/`prod`) albo nazwy modelu -- `ref()` (i `source()`) automatycznie podstawia WŁAŚCIWĄ, w pełni kwalifikowaną nazwę dla aktualnego środowiska, więc kod pozostaje poprawny nawet po takich zmianach.

<a id="sec4"></a>
## 4. Materializacje: view, table, incremental, ephemeral

**Materializacja** decyduje, JAK fizycznie dbt zapisuje wynik modelu w hurtowni:

- **`view`** (domyślna) -- zwykły widok SQL: żadnych danych nie kopiuje, zapytanie
  liczy się na nowo przy KAŻDYM odczycie. Szybkie do zbudowania, ale wolniejsze przy
  odczycie dla złożonych zapytań.
- **`table`** -- fizyczna tabela: dbt liczy zapytanie RAZ (przy `dbt run`) i zapisuje
  wynik. Szybszy odczyt, ale dane są nieaktualne aż do następnego uruchomienia.
- **`incremental`** -- jak `table`, ale przy KOLEJNYCH uruchomieniach dbt PRZETWARZA
  TYLKO nowe/zmienione wiersze zamiast całej tabeli od zera -- kluczowe dla dużych
  tabel faktów (poznamy szczegółowo w Module 5).
- **`ephemeral`** -- model w ogóle NIE jest materializowany jako osobny obiekt w
  hurtowni -- dbt "wkleja" jego SQL jako CTE (`WITH ...`) do modeli, które go
  używają przez `ref()`. Przydatne dla drobnych kroków pośrednich, które nie muszą
  istnieć jako osobna tabela.

Materializację ustawia się w bloku `{{ config(...) }}` na GÓRZE pliku modelu.

In [ ]:
model_kod = '''
{{ config(materialized='table') }}

select
    status,
    count(*) as liczba_zamowien
from {{ ref('stg_zamowienia') }}
group by status
'''
with open(f"{DBT_DIR}/models/podsumowanie_statusow.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "podsumowanie_statusow")


In [ ]:
con = duckdb.connect(DUCKDB_PATH)
# sprawdzamy, ze to FIZYCZNA TABELA (table), nie widok (view)
typ_obiektu = con.execute(
    "SELECT table_type FROM information_schema.tables WHERE table_name = 'podsumowanie_statusow'"
).fetchone()
print(f"Typ obiektu w DuckDB: {typ_obiektu[0]}")
print(con.execute("SELECT * FROM main.podsumowanie_statusow ORDER BY liczba_zamowien DESC").fetchdf())
con.close()


<a id="sec5"></a>
## 5. `dbt run`, `dbt build`, `dbt compile`

- **`dbt run`** -- buduje modele (w poprawnej kolejności zależności). Flaga
  `--select nazwa_modelu` ogranicza uruchomienie do JEDNEGO modelu (albo
  `--select nazwa_modelu+` -- ten model ORAZ wszystko, co go używa dalej).
- **`dbt build`** -- robi to, co `dbt run`, ORAZ dodatkowo uruchamia testy danych
  (`dbt test`, Moduł 5) i snapshoty -- jedna komenda za wszystko, częsta w CI/CD.
- **`dbt compile`** -- NIE wykonuje nic w hurtowni, tylko pokazuje, jak wygląda
  finalny, "rozwinięty" SQL PO podstawieniu `ref()`/`source()`/Jinja -- nieoceniony
  do debugowania.

In [ ]:
dbt("compile", "--select", "zamowienia_wzbogacone")

# skompilowany SQL (z podstawionymi ref()/source()) trafia do target/compiled/...
import glob
sciezki = glob.glob(f"{DBT_DIR}/target/compiled/**/zamowienia_wzbogacone.sql", recursive=True)
if sciezki:
    with open(sciezki[0]) as f:
        print(f.read())


> ⚡ **Skompilowany SQL to ZWYKŁY SQL**
>
> To, co widzisz powyżej, to czysty SQL bez żadnej składni dbt/Jinja -- dokładnie to, co dbt wysyła do silnika bazy danych. Gdy coś nie działa, `dbt compile` (albo podgląd plików w `target/compiled/`) to pierwsze miejsce do sprawdzenia -- czy wygenerowany SQL w ogóle robi to, co zamierzałeś.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- strukturę projektu dbt (`dbt_project.yml`, `profiles.yml`, `models/`, `sources.yml`),
- `source()` -- odwołanie do surowych, zewnętrznie załadowanych tabel,
- `ref()` -- odwołanie do INNYCH modeli dbt, budujące automatycznie graf zależności,
- cztery materializacje: `view`, `table`, `incremental`, `ephemeral`,
- `dbt run`, `dbt build`, `dbt compile` i flagę `--select`.

### 📝 Ćwiczenie 1: Model staging dla produktów

Napisz model `stg_produkty.sql`, który wybiera wszystkie kolumny z `source('raw_dane', 'produkty')` i dodaje kolumnę `cena_z_vat` (cena * 1.23). Uruchom go przez `dbt run --select stg_produkty` i sprawdź wynik w DuckDB.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
model_kod = '''
select
    produkt_id,
    kategoria,
    cena,
    round(cena * 1.23, 2) as cena_z_vat
from {{ source('raw_dane', 'produkty') }}
'''
with open(f"{DBT_DIR}/models/stg_produkty.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_produkty")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.stg_produkty LIMIT 5").fetchdf())
con.close()
```

</details>

### 📝 Ćwiczenie 2: Model z ref() liczący przychód per kategoria

Napisz model `przychod_kategorie.sql`, który łączy `ref('stg_produkty')` z `source('raw_dane', 'pozycje_zamowien')` i liczy sumę `ilosc * cena_jednostkowa * (1 - rabat)` PER `kategoria`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
model_kod = '''
select
    p.kategoria,
    round(sum(poz.ilosc * poz.cena_jednostkowa * (1 - poz.rabat)), 2) as przychod
from {{ source('raw_dane', 'pozycje_zamowien') }} poz
inner join {{ ref('stg_produkty') }} p on poz.produkt_id = p.produkt_id
group by p.kategoria
order by przychod desc
'''
with open(f"{DBT_DIR}/models/przychod_kategorie.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "przychod_kategorie")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.przychod_kategorie").fetchdf())
con.close()
```

</details>

### 📝 Ćwiczenie 3: Zmień materializację na table

Dodaj `{{ config(materialized='table') }}` do modelu `przychod_kategorie` z ćwiczenia 2, uruchom ponownie i sprawdź w `information_schema.tables`, że `table_type` zmienił się z `VIEW` na `BASE TABLE`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
model_kod = '''
{{ config(materialized='table') }}

select
    p.kategoria,
    round(sum(poz.ilosc * poz.cena_jednostkowa * (1 - poz.rabat)), 2) as przychod
from {{ source('raw_dane', 'pozycje_zamowien') }} poz
inner join {{ ref('stg_produkty') }} p on poz.produkt_id = p.produkt_id
group by p.kategoria
order by przychod desc
'''
with open(f"{DBT_DIR}/models/przychod_kategorie.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "przychod_kategorie")

con = duckdb.connect(DUCKDB_PATH)
typ = con.execute(
    "SELECT table_type FROM information_schema.tables WHERE table_name = 'przychod_kategorie'"
).fetchone()
print(f"Typ obiektu: {typ[0]}")
con.close()
```

</details>

> 🔥 **Wyzwanie: łańcuch trzech modeli**
>
> Zbuduj TRZY modele tworzące łańcuch przez `ref()`: (1) `stg_pozycje.sql` -- czyszczący `source('raw_dane', 'pozycje_zamowien')` (odrzuca wiersze z ujemną `ilosc`), (2) `wartosc_zamowien.sql` -- liczy `wartosc` PER `zamowienie_id` na podstawie `ref('stg_pozycje')`, (3) `top10_zamowien.sql` -- łączy `ref('wartosc_zamowien')` z `ref('stg_klienci')` (z sekcji 2) i zwraca 10 zamówień o najwyższej wartości wraz z miastem klienta. Uruchom `dbt run` BEZ flagi `--select` i sprawdź, że dbt sam ustalił poprawną kolejność.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
model_kod = '''
select *
from {{ source('raw_dane', 'pozycje_zamowien') }}
where ilosc > 0
'''
with open(f"{DBT_DIR}/models/stg_pozycje.sql", "w") as f:
    f.write(model_kod)

model_kod = '''
select
    zamowienie_id,
    round(sum(ilosc * cena_jednostkowa * (1 - rabat)), 2) as wartosc
from {{ ref('stg_pozycje') }}
group by zamowienie_id
'''
with open(f"{DBT_DIR}/models/wartosc_zamowien.sql", "w") as f:
    f.write(model_kod)

model_kod = '''
select
    w.zamowienie_id,
    z.klient_id,
    k.miasto,
    w.wartosc
from {{ ref('wartosc_zamowien') }} w
inner join {{ ref('stg_zamowienia') }} z on w.zamowienie_id = z.zamowienie_id
inner join {{ ref('stg_klienci') }} k on z.klient_id = k.klient_id
order by w.wartosc desc
limit 10
'''
with open(f"{DBT_DIR}/models/top10_zamowien.sql", "w") as f:
    f.write(model_kod)

dbt("run")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.top10_zamowien").fetchdf())
con.close()
```

Zwroc uwage w logach 'dbt run': mimo ze nie podalismy --select, dbt automatycznie uszeregowal modele w kolejnosci stg_pozycje -> wartosc_zamowien -> top10_zamowien (plus juz istniejace stg_klienci/stg_zamowienia), bo taka kolejnosc wynika z grafu ref().

</details>

## Co dalej?

W **Module 5** dodajemy testy danych (`dbt test`), dokumentację (`dbt docs`), snapshoty
oraz zaawansowane wzorce (macros, incremental strategies, seeds) -- czyniąc nasze modele
gotowymi na produkcyjne użycie.